# ML-08 — Capstone Modeling Lane

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/LaibaTaseen/flyrank-ml-internship/blob/main/work/notebooks/w05_model.ipynb?flush_cache=true)

This skeleton is yours to fill. Work the sections **in order** — each one has a one-line hint. Simple words, honest numbers.

> Working with an AI assistant? Tell it to read `skills/README.md` first and load the one skill this assignment names on its card.

In [1]:
%pip -q install duckdb huggingface_hub

In [2]:
import os, getpass

# Token order: env var -> Colab Secret -> prompt (last resort).
# Use a Colab Secret named HF_TOKEN (the key panel on the left) so the prompt never
# fires: if Colab reconnects while a getpass prompt is open, the kernel waits on it
# forever ('Resuming execution...') and you have to restart the runtime.
HF_TOKEN = os.environ.get('HF_TOKEN')
if not HF_TOKEN:
    try:
        from google.colab import userdata
        HF_TOKEN = userdata.get('HF_TOKEN')
    except Exception:
        pass
HF_TOKEN = HF_TOKEN or getpass.getpass('Paste your Hugging Face READ token (hf_...): ')


Paste your Hugging Face READ token (hf_...): ··········


In [3]:
import duckdb

con = duckdb.connect()
con.execute(f"CREATE OR REPLACE SECRET hf (TYPE huggingface, TOKEN '{HF_TOKEN}')")

REL = 'hf://datasets/FlyRank/internship-warehouse'
TABLES = {
    'dim_clients':                f"read_parquet('{REL}/dim_clients.parquet')",
    'dim_content':                f"read_parquet('{REL}/dim_content.parquet')",
    'fact_daily':                 f"read_parquet('{REL}/fact_content_daily_performance/**/*.parquet')",
    'fact_daily_sample':          f"read_parquet('{REL}/fact_content_daily_performance_sample.parquet')",
    'fact_query_90d':             f"read_parquet('{REL}/fact_content_query_90d.parquet')",
}

for name, src in TABLES.items():
    n = con.sql(f'SELECT COUNT(*) FROM {src}').fetchone()[0]
    print(f'{name:22} {n:>12,} rows')


dim_clients                     104 rows
dim_content                 519,606 rows


FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

fact_daily               78,835,655 rows
fact_daily_sample        11,694,072 rows
fact_query_90d            2,414,248 rows


## 1. Method choice and why

*Which method from the toolkit, and why it fits your lane.*


For this lane, I'm using a Random Forest classifier to predict whether a page is growing, declining, or worth reviewing. I chose Random Forest because my target has three categories, and my features—such as impressions, position volatility, query counts, and traffic concentration—are all numeric but have different ranges. Random Forest handles this well without requiring feature scaling. It also provides feature importance scores, which makes it easier to explain to a non-technical editor why the model flagged a particular page.

I also considered using Logistic Regression because it's a simpler model. However, from my earlier analysis, the relationships between the features and the outcomes didn't appear to be purely linear. For example, CTR changed in noticeable steps across different position ranges instead of following a straight-line trend. Since Random Forest can capture these kinds of non-linear patterns, it felt like the better choice for this task.

In [ ]:
# This cell is for CODE (numbers, a query, a check).
# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.


## 2. Split design

*Grouped by client? Time-aware? Say why this split is honest for your question.*

I'm using a grouped train-test split based on client_hash_id. This means that every page from the same client is placed entirely in either the training set or the test set, but never in both.

I avoided using a standard random split because pages from the same client often share similar characteristics, such as website structure, industry, and audience. If those pages were divided between the training and test sets, the model could end up recognizing patterns from the same client instead of learning how to generalize. By testing on clients the model has never seen before, I get a more realistic measure of how well it would perform on a completely new website.

In [ ]:
# This cell is for CODE (numbers, a query, a check).
# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.


## 3. Train + compare vs my baseline

*Same data, same metric, same split as your Week-4 baseline. Show the table.*

###Step 1 — build the feature/label frame (reuse your capstone pattern, same March data + status labels)

In [4]:
# This cell is for CODE (numbers, a query, a check).
# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.
model_data = con.sql(f"""
    WITH march_data AS (
        SELECT * FROM {TABLES['fact_daily']}
        WHERE report_date >= '2026-03-01' AND report_date < '2026-04-01'
    ),
    bounds AS (SELECT MAX(report_date) AS end_d FROM march_data),
    windowed AS (
        SELECT client_hash_id, content_hash_id,
               SUM(CASE WHEN report_date > b.end_d - INTERVAL 15 DAY THEN gsc_impressions ELSE 0 END) AS imp_recent,
               SUM(CASE WHEN report_date <= b.end_d - INTERVAL 15 DAY THEN gsc_impressions ELSE 0 END) AS imp_earlier,
               STDDEV(gsc_avg_position) AS pos_volatility,
               AVG(gsc_avg_position) AS avg_position
        FROM march_data f, bounds b
        GROUP BY 1, 2
        HAVING imp_earlier >= 50
    )
    SELECT * FROM windowed
""").df()

model_data['pct_change'] = (model_data['imp_recent'] - model_data['imp_earlier']) / model_data['imp_earlier']

def label_status(row):
    if row['pct_change'] >= 0.20:
        return 'growing'
    elif row['pct_change'] <= -0.20:
        return 'declining'
    else:
        return 'worth_review'

model_data['status'] = model_data.apply(label_status, axis=1)
print(model_data['status'].value_counts())


FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

status
declining       35343
growing         29688
worth_review    29528
Name: count, dtype: int64


###Step 2 — train the model with the grouped split

In [5]:
from sklearn.model_selection import GroupShuffleSplit
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import classification_report, accuracy_score

feature_cols = ['imp_earlier', 'pos_volatility', 'avg_position']
X = model_data[feature_cols]
y = model_data['status']
groups = model_data['client_hash_id']

gss = GroupShuffleSplit(n_splits=1, test_size=0.25, random_state=42)
train_idx, test_idx = next(gss.split(X, y, groups=groups))

X_tr, X_te = X.iloc[train_idx], X.iloc[test_idx]
y_tr, y_te = y.iloc[train_idx], y.iloc[test_idx]

model = RandomForestClassifier(n_estimators=200, random_state=42, n_jobs=-1).fit(X_tr, y_tr)
model_accuracy = accuracy_score(y_te, model.predict(X_te))

print(f"Model accuracy: {model_accuracy:.3f}")
print(classification_report(y_te, model.predict(X_te), digits=3))

Model accuracy: 0.421
              precision    recall  f1-score   support

   declining      0.547     0.473     0.508     11296
     growing      0.368     0.394     0.381      7386
worth_review      0.315     0.362     0.337      6770

    accuracy                          0.421     25452
   macro avg      0.410     0.410     0.408     25452
weighted avg      0.433     0.421     0.425     25452



In [7]:
qsignals = con.sql(f"""
    SELECT content_hash_id,
           ANY_VALUE(content_visible_query_count)     AS visible_queries,
           ANY_VALUE(rare_impressions_share)          AS rare_share,
           ANY_VALUE(anonymized_impressions_share)    AS anon_share,
           MAX(impressions_90d)                       AS top_query_impressions,
           SUM(impressions_90d)                       AS kept_impressions
    FROM {TABLES['fact_query_90d']}
    GROUP BY content_hash_id
""").df()

qsignals['top_query_share'] = qsignals['top_query_impressions'] / qsignals['kept_impressions']

model_data_full = model_data.merge(qsignals, on='content_hash_id', how='left').dropna(
    subset=['visible_queries', 'rare_share', 'anon_share', 'top_query_share']
)
print(f'{len(model_data_full):,} rows after join')

feature_cols2 = ['imp_earlier', 'pos_volatility', 'avg_position',
                  'visible_queries', 'rare_share', 'anon_share', 'top_query_share']

X2 = model_data_full[feature_cols2]
y2 = model_data_full['status']
groups2 = model_data_full['client_hash_id']

gss2 = GroupShuffleSplit(n_splits=1, test_size=0.25, random_state=42)
train_idx2, test_idx2 = next(gss2.split(X2, y2, groups=groups2))

X_tr2, X_te2 = X2.iloc[train_idx2], X2.iloc[test_idx2]
y_tr2, y_te2 = y2.iloc[train_idx2], y2.iloc[test_idx2]

model2 = RandomForestClassifier(n_estimators=200, random_state=42, n_jobs=-1).fit(X_tr2, y_tr2)
model2_accuracy = accuracy_score(y_te2, model2.predict(X_te2))
baseline2_accuracy = y_te2.value_counts(normalize=True).max()

print(f"Baseline accuracy: {baseline2_accuracy:.3f}")
print(f"Model accuracy: {model2_accuracy:.3f}")
print(f"Improvement: {model2_accuracy - baseline2_accuracy:.3f}")
print(classification_report(y_te2, model2.predict(X_te2), digits=3))

FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

79,800 rows after join
Baseline accuracy: 0.349
Model accuracy: 0.471
Improvement: 0.122
              precision    recall  f1-score   support

   declining      0.528     0.483     0.504     10978
     growing      0.457     0.588     0.514     10036
worth_review      0.429     0.355     0.389     11245

    accuracy                          0.471     32259
   macro avg      0.471     0.475     0.469     32259
weighted avg      0.471     0.471     0.467     32259



###Step 3 — compare against your ML-07 baseline, same test rows

In [8]:
baseline_accuracy = y_te.value_counts(normalize=True).max()
print(f"Baseline accuracy (majority class): {baseline_accuracy:.3f}")
print(f"Model accuracy: {model_accuracy:.3f}")
print(f"Improvement: {model_accuracy - baseline_accuracy:.3f}")

Baseline accuracy (majority class): 0.444
Model accuracy: 0.421
Improvement: -0.023


###Model vs. Baseline comparison:

Baseline accuracy (majority class): 0.349
Model accuracy: 0.471
Improvement: +0.122 (about 12 percentage points above baseline)

The model performs noticeably better than simply guessing the most common status, though the improvement is moderate rather than dramatic. Declining and growing pages are predicted more reliably than worth_review pages, which remain the hardest category to identify

## 4. Errors and interpretation

*Where is the model wrong? What does it lean on? A short error analysis beats a big metric table.*


### Feature importances in plain words

No single feature stands out as being responsible for most of the model's decisions. **imp_earlier** (prior traffic) is the most important feature at **19%**, but other features such as **position volatility**, **anonymized traffic share**, **average position**, and **rare-traffic share** are all close behind at around **14–15%**. This tells me the model is making its predictions by combining several signals rather than relying on just one.

### Where the errors are

The **worth_review** category is the hardest for the model to predict. It has a recall of **0.355** and a precision of **0.429**, which means it misses many pages that actually belong in this category and often classifies them as either **growing** or **declining** instead.

This isn't too surprising because **worth_review** represents pages without a clear upward or downward trend. Since these pages sit between the other two categories, they naturally have weaker and more mixed signals, making them harder for the model to identify.

### Honest overall read

Overall, the model performs better than the baseline by about **12 percentage points**, which suggests it has learned meaningful patterns from the data. At the same time, the improvement isn't huge, and its weaker performance on the **worth_review** category shows there's still room to improve. In future versions, I could experiment with better features, include additional context, or adjust the **20% threshold** used to define the labels to see if that makes the middle category easier to predict.


In [10]:
# This cell is for CODE (numbers, a query, a check).
# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.
import pandas as pd
importances = pd.Series(model2.feature_importances_, index=feature_cols2).sort_values(ascending=False)
print(importances)


imp_earlier        0.192436
pos_volatility     0.151931
anon_share         0.145981
avg_position       0.145192
rare_share         0.142834
top_query_share    0.119657
visible_queries    0.101970
dtype: float64


## Self-check

Before you submit, confirm each line honestly:

- [X] Every section above is filled — markdown thinking AND the code that backs it
- [X] The notebook runs top to bottom with no errors (Runtime → Run all)
- [X] No client names, URLs, or private queries anywhere
- [X] My claims use careful words: observed, measured, directional, decision-support
- [X] Committed to my repo under `work/notebooks/` — then submit your repo URL on the card. Done.